# Inference Visualizer
## Developed by Moose Abou-Harb on behalf of Paccar Inc

In [1]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np

import json
import pathlib
import math

store_folder_path = pathlib.Path.cwd().parent / "test_data"
gt_file = store_folder_path / "all_objects_ground_truth.json"

earth_radius = 6378137.0
deg_to_rad = math.pi / 180

modalities = ["camera", "lidar", "radar"]

def main():
    #Load up the data from the sim
    gt_data = {}
    inferences = {}
    try:
        gt_data = try_load_json(gt_file)
        for modality in modalities:
            modality_path = store_folder_path / f"{modality}_sim_results.json"
            inferences[modality] = try_load_json(modality_path)["inferences"]
    except Exception as e:
        print("Failed to load sim data!")
        return

    #Get the origin
    origin = gt_data["start_pos"]

    #Shove that data into a dataframe
    main_df = pd.DataFrame()
    for modality in modalities:
        new_df = pd.DataFrame(inferences[modality])
        new_df["modality"] = modality
        main_df = pd.concat([main_df, new_df])

    #Expand lat/long/alt into local coords
    main_df[["x", "y", "z"]] = main_df.apply(
        lambda row : geo_to_local(origin, [row["latitude"], row["longitude"], row["altitude"]]), 
        axis=1,
        result_type="expand"
    )

    #Create an origin dataframe
    origin_df = pd.DataFrame([{
        "timestamp" : 0,
        "class" : "origin",
        "latitude" : origin[0],
        "longitude" : origin[1],
        "altitude" : origin[2],
        "dimensions" : [0, 0, 0],
        "obj_id" : -1,
        "modality" : "origin",
        "x" : 0,
        "y" : 0,
        "z" : 0
    }])

    #Merge in the origin point
    main_df = pd.concat([origin_df, main_df])

    #Expand dimensions into individual columns
    main_df[['dx', 'dy', 'dz']] = pd.DataFrame(main_df['dimensions'].tolist(), index=main_df.index)
    main_df = main_df.drop("dimensions", axis=1)
    main_df = main_df.sort_values(by="obj_id")
    
    display(main_df.head(30))

    fig = px.scatter_3d(main_df, x='x', y='y', z='z', opacity=0.7, color="modality")

    fig.update_traces(marker=dict(size=5))

    fig.show()

def try_load_json(fp: str) -> dict:
    try:
        with open(fp, 'r') as file:
            data = json.load(file)
            return data
    except FileNotFoundError:
        print(f"Failed to load file: {fp}")
    except json.JSONDecodeError:
        print(f"Selected file contains illegal JSON: {fp}")
    except Exception as e:
        print(f"Something went wrong while loading: {fp}, {e}")

def geo_to_local(origin, target):
    d_lat = target[0] - origin[0]
    d_lon = target[1] - origin[1]
    d_alt = target[2] - origin[2]
    y_meters = d_lat * deg_to_rad * earth_radius

    radius_at_lat = earth_radius * math.cos(origin[0] * deg_to_rad)
    x_meters = d_lon * deg_to_rad * radius_at_lat

    return [x_meters, y_meters, d_alt]

if __name__ == "__main__":
    main()

,timestamp,class,latitude,longitude,altitude,obj_id,modality,x,y,z,dx,dy,dz
0,0.000000,origin,47.747343,1.309586,4286.362907,-1,origin,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
0,86.755080,traffic_cone,47.747362,1.309743,4290.078696,0,camera,11.720664,2.099615,3.715789,0.546124,0.372332,0.944583
0,86.710571,vehicle,47.747362,1.309741,4290.076051,0,lidar,11.589308,2.189650,3.713144,0.562108,0.365285,0.952329
0,86.789638,vehicle,47.747359,1.309747,4290.134427,0,radar,12.075953,1.784792,3.771520,0.636348,0.411013,0.936625
1,86.726858,traffic_cone,47.747373,1.309559,4283.787557,1,lidar,-2.048020,3.326827,-2.575350,0.922945,0.148690,0.588724
1,86.759015,traffic_cone,47.747371,1.309558,4283.784258,1,camera,-2.109335,3.171128,-2.578649,0.893955,0.164142,0.633120
1,86.722642,vehicle,47.747371,1.309564,4283.752136,1,radar,-1.684684,3.175437,-2.610771,0.994786,0.191562,0.547822
2,86.765022,pedestrian,47.747267,1.309405,4286.765195,2,lidar,-13.574034,-8.384126,0.402289,0.675424,0.866378,0.851282
2,86.731474,pedestrian,47.747266,1.309407,4286.806587,2,camera,-13.421681,-8.505921,0.443680,0.701334,0.864859,0.819487
2,86.758770,pedestrian,47.747269,1.309403,4286.755608,2,radar,-13.681010,-8.255764,0.392701,0.815284,0.850457,0.832947
